<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Machine-Learning/20-ml-systems-research-practice.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回机器学习总览](Machine Learning.html)


## **机器学习系统与研究实践**

拟合完成的模型是一个数学对象；**机器学习系统**则是把观测转化为行动，并能在数据、用户、基础设施和组织决策不断变化时继续可靠运行的完整机制。它包括数据收集、标签构造、特征计算、训练、评估、工件存储、在线服务、监控、降级与回退、人类复核，以及历史预测所产生的反馈。一个模型在统计上可能很强，围绕它构造的系统却仍然可能不可靠。

这一区别会改变核心问题。模型开发询问：

> 哪个函数 $f_\theta(x)$ 能在恰当的评估分布上良好预测目标？

系统开发询问：

> 在怎样的数据、运行和决策契约下，完整流水线能够在不违反约束的前提下创造净价值？

一种有用的抽象是带约束的期望效用：

$$
\max_{\pi,\,f_\theta}
\quad
\mathbb{E}\left[
B\bigl(Y,\pi(f_\theta(X))\bigr)
-C_{\text{decision}}
-C_{\text{operation}}
-C_{\text{harm}}
\right]
$$

并满足如下要求：

$$
\text{latency}_{p99}\le L_{\max},
\qquad
\text{availability}\ge A_{\min},
\qquad
\text{memory}\le M_{\max},
\qquad
g_k(\text{slice metrics})\le \tau_k.
$$

其中，$f_\theta$ 产生分数或预测，$\pi$ 是把预测转化为行动的**决策策略**，它可能包含阈值、拒绝预测区间、容量限制或人工复核。目标函数同时计入收益与多种成本。约束并不是用于报告的装饰性指标：即使某个候选模型的平均测试分数最高，只要违反硬性的安全、延迟、隐私或资源要求，它就是不可行方案。

下面这张官方图揭示了机器学习系统的第一条重要规律：生产中的模型代码被数据验证、配置、资源管理、服务基础设施、监控和流程工具所包围。

<div class="diagram-scroll">

![生产机器学习系统包含大量模型代码以外的组件。](assets/google-production-ml-system-official.png){fig-alt="Google 图中，ML 模型代码只是较小组件，周围还有数据收集、验证、特征提取、配置、监控、服务和资源管理。"}

</div>

*图片来源：[Google Machine Learning Crash Course：Production ML systems](https://developers.google.com/machine-learning/crash-course/production-ml-systems)，采用 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) 许可。*

模型的离线指标可能因为多种原因无法代表系统质量：

- 训练标签只是实际结果的代理变量；
- 服务路径计算出的特征不同于训练路径；
- p99 延迟尖峰恰好让最繁忙用户群体的请求超时；
- 系统采取的行动改变了哪些标签能够被观察到；
- 新模型提高平均准确率，却使内存或成本超出限制；
- 标签数周后才到达，性能衰减在此期间无法被直接观察；
- 一项看似成功的实验无法根据代码、数据和配置复现。

因此，正确性的单位是**端到端契约**，而不是孤立的估计器。

<div class="diagram-scroll">

![机器学习生命周期从问题定义经过运行，再由生产证据闭环返回。](assets/ml-system-lifecycle.svg){fig-alt="五个阶段依次表示问题定义、数据、实验、发布和运行，生产证据再反馈到下一轮问题定义。"}

</div>

这个生命周期会产生多种不同工件：

| 阶段 | 核心问题 | 持久化工件 |
|---|---|---|
| 问题定义 | 要改善什么决策、服务谁、受哪些约束？ | 决策契约与基线 |
| 数据 | 历史预测时刻能够知道什么，标签如何产生？ | 数据契约、血缘与验证报告 |
| 实验 | 哪个受控比较支持所声称的改进？ | 运行记录、预测、置信度与消融实验 |
| 发布 | 候选方案能否被打包、测试、渐进发布并撤回？ | 版本化模型包与部署计划 |
| 运行 | 已部署的决策流程是否仍健康且有价值？ | 监控、事故、延迟标签评估与退役记录 |

<details>
<summary><strong>Python：在执行硬性系统约束后按效用评估候选方案</strong></summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Candidate:
    name: str
    true_positives: int
    false_positives: int
    false_negatives: int
    p99_latency_ms: float
    availability: float
    cost_per_1000: float
    worst_slice_recall: float


def assess(candidate: Candidate) -> dict:
    # The utility values belong to the decision context, not to the classifier.
    benefit = 8.0 * candidate.true_positives
    decision_cost = 2.0 * candidate.false_positives + 5.0 * candidate.false_negatives
    operating_cost = candidate.cost_per_1000 * 40.0  # expected daily volume in thousands
    utility = benefit - decision_cost - operating_cost

    checks = {
        "p99_latency": candidate.p99_latency_ms <= 80.0,
        "availability": candidate.availability >= 0.999,
        "slice_recall": candidate.worst_slice_recall >= 0.72,
    }
    return {
        "name": candidate.name,
        "utility": round(utility, 1),
        "feasible": all(checks.values()),
        "failed_constraints": [name for name, passed in checks.items() if not passed],
    }


candidates = [
    Candidate("heuristic", 620, 150, 210, 4, 0.9999, 0.02, 0.74),
    Candidate("compact_model", 710, 170, 120, 24, 0.9997, 0.35, 0.78),
    Candidate("large_ensemble", 745, 155, 85, 132, 0.9987, 4.80, 0.81),
]

for result in map(assess, candidates):
    print(result)
```

</details>

大型集成模型拥有最好的预测计数，但在既定延迟和可用性契约下并不是合法的发布候选。这并不意味着应该隐藏它的结果；它为下一步工程问题提供了证据：能否压缩模型、预计算输出、缩短特征路径、经利益相关者批准后放宽决策时限，或者继续保留紧凑模型。

**对比总结。** Notebook 中的模型针对数据集优化；机器学习系统针对决策流程优化，并需要在变化中持续维护。模型指标仍然必不可少，但它只是更完整可靠性论证中的一层证据。


### **问题定义与系统边界**

问题定义把“预测流失”或“检测欺诈”这样的模糊请求转化为可证伪、可运行的契约。契约必须明确**决策单位**、**预测时刻**、**目标时间范围**、**行动**、**受影响总体**、**标签机制**、**基线策略**、**处理容量**和**错误成本**。只要其中一项含糊，两个团队就可能分别为不同问题构建高准确率模型，却误以为自己解决的是同一个问题。

<div class="diagram-scroll">

![问题定义把用户和观测连接到决策、结果与证据。](assets/problem-framing-contract.svg){fig-alt="流程从用户状态经过预测、决策和结果到达证据，强调预测指标必须连接到实际行动。"}

</div>

#### **用户、决策、约束与成功指标**

严谨的问题定义应当在选择模型前回答下列问题：

| 契约要素 | 需要精确回答的问题 | 常见失败 |
|---|---|---|
| 决策单位 | 一行代表用户、账户、会话、交易、图像还是时间窗口？ | 同一实体的多行跨越数据划分而泄漏 |
| 预测时刻 | 预测究竟必须在哪个时刻可用？ | 使用决策时刻之后计算的特征 |
| 目标与范围 | 要预测什么事件，预测多长时间内的事件？ | 混合短期与长期标签 |
| 行动 | 分数高或低时，系统会改变什么？ | 构建无人能够采取行动的预测 |
| 容量 | 下游流程最多能够处理多少案例？ | 优化出的阈值超过复核容量 |
| 错误与伤害 | 假阳性、假阴性、拒绝预测和延迟分别有什么成本？ | 把所有错误视为等价 |
| 基线 | 没有模型时当前流程怎样运行？ | 在没有有意义比较对象时声称提升 |
| 反馈 | 行动是否改变曝光、行为或标签可观测性？ | 在选择性可观测结果上训练 |
| 约束 | 哪些延迟、可用性、内存、隐私与切片要求是硬限制？ | 实验结束后才发现最优模型不可部署 |

**成功指标**既要足够接近真实目标，能够指导改进，也要足够稳定，能够反复测量。通常不存在同时满足两者的单一指标，因此系统会组合使用：

- **主要决策指标**，例如期望节省金额、容量内正确排在前列的案例数，或成功解决所需时间；
- **模型诊断指标**，例如对数损失、AUROC、校准度、固定工作量下的召回率或排序质量；
- **护栏指标**，例如 p99 延迟、子群体假阴性率、投诉率、内存或成本；
- **领先指标**，能够快速获得，但与最终结果的关系并不完美；
- **滞后结果**，可信度更高，但到达较晚。

必须区分**目标函数**、**评估指标**和**约束**。训练目标是算法优化的可微量；评估指标估计预测的某项性质；系统目标表示决策价值；硬约束定义可行性。改善其中一项并不能保证其他各项同时改善。

#### **何时不应使用机器学习**

问题包含数据，并不代表机器学习就一定合适。出现下列情况时，应优先考虑确定性规则、查询、优化算法或人工流程：

- 期望行为能够被精确指定，并且很少变化；
- 代表性样本太少，或不存在可信标签机制；
- 错误不可接受，同时又无法被界定、检测、复核或撤销；
- 无论预测多准确，后续行动都无法改变；
- 环境变化速度快于标签到达和重训练速度；
- 真正需求是因果干预、数据库检索、约束满足或异常调查，而不是预测；
- 简单启发式已经满足决策与运行契约；
- 组织无法负责监控、事故响应、访问控制和系统退役。

正确比较不是“机器学习与什么都不做”，而是**机器学习与最佳的可维护非 ML 方案**。固定规则可能更透明、可测试、便宜且稳定；反过来，一组不断增长且相互影响的手写规则，也可能比根据反馈训练的监督模型更难维护。

<details>
<summary><strong>Python：在统一决策价值与复核容量下比较不同策略</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n = 5000

# Simulated latent event probability and binary outcome.
risk = rng.beta(2.0, 8.0, size=n)
y = rng.binomial(1, risk)

# Three policies: no review, a cheap heuristic, and an ML score.
heuristic_score = np.clip(risk + rng.normal(0, 0.18, n), 0, 1)
model_score = np.clip(risk + rng.normal(0, 0.09, n), 0, 1)

capacity = 500
benefit_if_caught = 12.0
review_cost = 1.5


def top_capacity_value(name, score):
    selected = np.argsort(score)[-capacity:]
    caught = int(y[selected].sum())
    value = benefit_if_caught * caught - review_cost * capacity
    return {"policy": name, "reviewed": capacity, "events_caught": caught, "value": value}


print({"policy": "no_review", "reviewed": 0, "events_caught": 0, "value": 0.0})
print(top_capacity_value("heuristic", heuristic_score))
print(top_capacity_value("ml_model", model_score))
```

</details>

这个例子让所有策略面对相同的下游容量。使用 $0.5$ 之类的固定阈值会很武断，因为团队只能复核 500 个案例。因此，合适的系统指标是**容量内捕获的事件数**，随后再减去复核成本计算净价值。

### **基线优先的开发方式**

基线是已经实现的比较对象，用于检验项目是否真正学到了有用信息。它还能在模型复杂度掩盖问题前暴露标签错误、数据划分泄漏、指标错误和集成成本。良好的基线阶梯要求每增加一级复杂度，都给出相应证据。

<div class="diagram-scroll">

![基线阶梯要求每次增加复杂度都给出受控证据。](assets/baseline-evidence-ladder.svg){fig-alt="阶梯从虚拟和启发式基线，经过经典与候选模型，最终到达可部署系统。"}

</div>

#### **虚拟、启发式与经典基线**

- **虚拟基线**预测类别比例、均值、中位数、随机排序或季节值，用于检查指标和数据划分是否合理。
- **启发式基线**编码现有领域知识或当前策略，往往才是真正的运行比较对象。
- **经典基线**使用透明且低成本的模型，例如正则化线性或逻辑回归、浅层树、近邻方法或简单时间序列方法。
- **系统基线**包括现有数据流水线、服务路径、延迟、成本和结果，防止纯模型比较忽略集成影响。

基线不应该被故意做弱。如果存在易于获得的更强且调优合理的基线，使用弱基线会夸大贡献。不同候选之间的超参数预算、预处理、数据访问和评估协议应尽量可比。

#### **消融实验与受控比较**

**消融实验**在保持其余协议不变时移除或替换一个组件。如果一个系统同时加入新特征组、损失项、采样器、模型模块和后处理规则，最终分数无法说明究竟哪个组件带来了提升。受控消融只改变一个因素，并重新执行完整评估。

对于成对测试样本，可以定义候选 $A$ 相对基线 $B$ 的逐样本差值：

$$
d_i=\ell\bigl(y_i,\hat y_i^{(B)}\bigr)
-\ell\bigl(y_i,\hat y_i^{(A)}\bigr).
$$

此时 $\bar d>0$ 支持 $A$。因为两个模型在相同样本上评估，不确定性估计也应保留配对关系，例如对样本或群组执行成对 bootstrap。当训练随机性不可忽略时，仍需要多个独立随机种子。

<details>
<summary><strong>Python：运行基线阶梯并执行受控特征消融</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.78, 0.22],
    class_sep=1.0,
    random_state=20,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=20
)

models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "forest": RandomForestClassifier(
        n_estimators=180, min_samples_leaf=10, n_jobs=-1, random_state=20
    ),
}

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    score = estimator.predict_proba(X_test)[:, 1]
    print(name, round(roc_auc_score(y_test, score), 4))

# Controlled ablation: same estimator, rows, seed, and metric; only the final
# four feature columns are removed.
full = RandomForestClassifier(
    n_estimators=180, min_samples_leaf=10, n_jobs=-1, random_state=20
).fit(X_train, y_train)
ablated = RandomForestClassifier(
    n_estimators=180, min_samples_leaf=10, n_jobs=-1, random_state=20
).fit(X_train[:, :-4], y_train)

full_auc = roc_auc_score(y_test, full.predict_proba(X_test)[:, 1])
ablated_auc = roc_auc_score(y_test, ablated.predict_proba(X_test[:, :-4])[:, 1])
print("feature_group_contribution", round(full_auc - ablated_auc, 4))
```

</details>

**对比总结。** 问题定义说明成功意味着什么；基线判断学习算法是否优于可信替代方案；消融实验识别哪个新增组件真正带来改进。三者共同防止一个较高的基准分数被误写成缺乏证据的系统结论。


### **数据与特征流水线**

数据流水线是把源事件转化为训练样本和服务输入的版本化流程。它的正确性既涉及时间，也涉及语义，而不仅仅是语法。两张表可以拥有完全相同的列和类型，却代表不同总体、时间窗口、单位或知识状态。

流水线至少应保留下列四类来源信息：

1. **来源：**源系统、负责人、收集机制与访问策略。
2. **语义：**实体、单位、时间戳含义、缺失值含义与有效范围。
3. **变换：**代码版本、参数、依赖和上游输入。
4. **时间：**事件时间、摄取时间、处理时间、预测时间与标签成熟时间。

**数据契约**把这些假设转化为可执行检查。它可以约束 schema、唯一性、空值策略、类别词表、时间顺序、新鲜度、数值范围和跨列不变量。契约测试应当在数据摄取、训练前和服务路径上分别运行。分布距离等统计检查能够补充契约，却不能替代语义验证：以美元计量的列可能悄悄变为以美分计量，同时仍保持数值类型和平滑分布。

#### **批处理数据与流式数据**

**批处理**操作有边界的数据集合，例如前一天的全部交易。它更容易重放、测试、聚合和修正，代价则是数据陈旧：决策发生时，特征可能已经落后数小时。

**流式处理**随事件到达更新状态。它支持低延迟决策和快速变化的特征，但会引入乱序、重复、迟到事件、检查点和状态恢复问题。因此，流处理需要明确的事件标识符、水位线、幂等更新和迟到数据策略。

必须区分三个时间戳：

- **事件时间：**真实世界事件发生的时间；
- **摄取时间：**平台接收到事件的时间；
- **处理时间：**计算过程处理事件的时间。

对事件 $e$，摄取延迟为

$$
\Delta_{\text{ingest}}(e)=t_{\text{ingest}}(e)-t_{\text{event}}(e).
$$

如果第 99 百分位延迟为两小时，那么在预测时刻，若没有明确的修正方法或水位线，就不能把“过去一小时交易数”视为完整特征。回填事件可以改善历史表，却也可能让历史数据不同于线上当时实际看到的不完整状态。

<details>
<summary><strong>Python：在训练前执行一个紧凑的数据契约</strong></summary>

```python
import pandas as pd

events = pd.DataFrame(
    {
        "event_id": ["e1", "e2", "e3", "e4"],
        "account_id": [101, 101, 102, 103],
        "event_time": pd.to_datetime(
            ["2026-04-01 09:00", "2026-04-01 10:00", "2026-04-01 09:30", "2026-04-01 10:10"],
            utc=True,
        ),
        "ingestion_time": pd.to_datetime(
            ["2026-04-01 09:03", "2026-04-01 10:07", "2026-04-01 09:31", "2026-04-01 10:12"],
            utc=True,
        ),
        "amount": [42.5, 18.0, 120.0, 7.5],
        "channel": ["web", "mobile", "web", "branch"],
    }
)


def validate_event_contract(frame: pd.DataFrame) -> list[str]:
    errors = []
    required = {
        "event_id", "account_id", "event_time",
        "ingestion_time", "amount", "channel",
    }
    missing = required - set(frame.columns)
    if missing:
        errors.append(f"missing columns: {sorted(missing)}")
        return errors

    if frame["event_id"].duplicated().any():
        errors.append("event_id must be unique")
    if frame[["account_id", "event_time", "amount"]].isna().any().any():
        errors.append("entity, event time, and amount cannot be null")
    if (frame["amount"] < 0).any() or (frame["amount"] > 1_000_000).any():
        errors.append("amount outside the documented range")
    if not set(frame["channel"]).issubset({"web", "mobile", "branch"}):
        errors.append("unknown channel")
    if (frame["ingestion_time"] < frame["event_time"]).any():
        errors.append("ingestion cannot precede event time")

    delay_minutes = (
        frame["ingestion_time"] - frame["event_time"]
    ).dt.total_seconds() / 60
    if delay_minutes.quantile(0.99) > 30:
        errors.append("p99 ingestion delay exceeds the 30-minute freshness contract")
    return errors


violations = validate_event_contract(events)
print("contract_status", "PASS" if not violations else "FAIL")
print("violations", violations)
```

</details>

#### **时间点正确性与标签构造**

训练行只能包含其历史预测发生时可知的信息。对于预测时间为 $t_i$ 的实体 $i$，时间为 $s$ 的特征事件只有在满足

$$
s\le t_i
$$

时才具备资格；在更真实的重放中，还必须保证它在 $t_i$ 之前已经被摄取并处理。**时间点正确的连接**会获取最近一条合格值，或聚合一个结束于 $t_i$ 的窗口。普通数据库连接如果直接使用表中的最新值，就可能把未来信息泄漏进所有历史行。

<div class="diagram-scroll">

![时间点连接重建历史预测时刻实际可获得的信息。](assets/point-in-time-data.svg){fig-alt="时间线上若干特征事件分布在预测时刻两侧，只有边界之前已经可知的事件可以进入训练行。"}

</div>

标签也需要同样的纪律。对于“30 天内违约”这样的目标，数据集必须：

- 定义结果窗口的起止位置；
- 排除或标记尚未完成 30 天观察窗口的样本；
- 区分真正负例与被截尾或缺失的结果；
- 记录某项行动是否阻止了事件被观察；
- 避免使用结果发生后的调查或处理过程所派生的特征。

<details>
<summary><strong>Python：使用 as-of join 构造时间点正确的特征</strong></summary>

```python
import pandas as pd

feature_events = pd.DataFrame(
    {
        "account_id": [1, 1, 1, 2, 2],
        "event_time": pd.to_datetime(
            ["2026-01-02", "2026-01-09", "2026-01-20", "2026-01-05", "2026-01-18"],
            utc=True,
        ),
        "balance": [100, 80, 20, 200, 160],
    }
)

prediction_rows = pd.DataFrame(
    {
        "row_id": ["r1", "r2", "r3"],
        "account_id": [1, 1, 2],
        "prediction_time": pd.to_datetime(
            ["2026-01-10", "2026-01-15", "2026-01-12"],
            utc=True,
        ),
    }
)

# merge_asof requires the time key to be globally sorted. The "by" key ensures
# that a row can use only events belonging to the same account.
joined = pd.merge_asof(
    prediction_rows.sort_values("prediction_time"),
    feature_events.sort_values("event_time"),
    left_on="prediction_time",
    right_on="event_time",
    by="account_id",
    direction="backward",
    allow_exact_matches=True,
)

assert (joined["event_time"] <= joined["prediction_time"]).all()
print(joined[["row_id", "account_id", "prediction_time", "event_time", "balance"]])
```

</details>

#### **训练与服务一致性**

训练-服务偏差是离线特征或预测流程与线上流程之间的差异，常见形式包括：

- **逻辑偏差：**两套实现计算了不同变换；
- **时间偏差：**训练看到完整回填历史，服务只看到最近的不完整数据；
- **数据源偏差：**离线与线上路径读取不同系统；
- **默认值偏差：**缺失值、未知类别或截断规则处理不一致；
- **版本偏差：**模型配错了特征 schema 或词表；
- **总体偏差：**服务总体不同于训练样本。

特征存储可以帮助集中管理定义、历史查询、在线物化和元数据，但不会自动保证正确。时间点连接、事件时间、变换归属、新鲜度和监控仍需明确设计。

<div class="diagram-scroll wide-diagram">

![Feast 架构区分变换、注册、存储以及在线或离线特征服务。](assets/feast-feature-store-architecture-official.png){fig-alt="Feast 官方架构图展示请求、流式和批量数据源经过变换与注册，为在线推理和离线训练提供特征。"}

</div>

*图片来源：[Feast 官方仓库架构图](https://github.com/feast-dev/feast#-architecture)，采用 [Apache License 2.0](https://github.com/feast-dev/feast/blob/master/LICENSE) 许可。*

最强的一致性模式是只定义一次变换，把它同时应用于离线和在线原始值，抽样记录服务时的特征向量，再由离线流水线重放。对于确定性变换，两侧应完全相等，或处于预先声明的数值容差内：

$$
\max_j\left|x^{\text{train}}_j-x^{\text{serve}}_j\right|\le \epsilon_j.
$$

对于有状态或近似特征，应比较时间戳、新鲜度、覆盖率和差值分布，而不是要求完全相等。

<details>
<summary><strong>Python：测试训练与服务共用的特征变换是否一致</strong></summary>

```python
import hashlib
import json
import numpy as np


FEATURE_VERSION = "customer_features_v3"


def transform_customer(raw: dict) -> dict:
    # This function is deliberately shared by training and serving.
    age = float(np.clip(raw["age"], 18, 100))
    income = max(float(raw.get("annual_income") or 0.0), 0.0)
    debt = max(float(raw.get("debt") or 0.0), 0.0)
    return {
        "age_scaled": (age - 18.0) / 82.0,
        "log_income": float(np.log1p(income)),
        "debt_to_income": debt / max(income, 1.0),
        "is_mobile": float(raw.get("channel") == "mobile"),
    }


def fingerprint(features: dict) -> str:
    canonical = json.dumps(features, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]


raw_record = {
    "age": 34,
    "annual_income": 72000,
    "debt": 18000,
    "channel": "mobile",
}

offline_features = transform_customer(raw_record)
served_log_features = transform_customer(raw_record)

differences = {
    name: abs(offline_features[name] - served_log_features[name])
    for name in offline_features
}
assert max(differences.values()) <= 1e-12

print("feature_version", FEATURE_VERSION)
print("fingerprint", fingerprint(offline_features))
print("maximum_absolute_difference", max(differences.values()))
```

</details>

**对比总结。** 批处理流水线强调可重放性和吞吐量；流式流水线强调新鲜度，但必须处理事件时间状态。特征存储协调定义与查询，却不能消除时间推理。数据契约发现结构违规，时间点连接避免历史泄漏，一致性测试发现训练与服务路径分叉。


### **可复现训练与实验追踪**

只有当另一轮运行能够恢复相关输入、流程和比较时，实验才具备可复现性。只保存最终模型远远不够：同一个模型类别如果在不同数据划分、依赖版本、预处理规则或随机种子下训练，就已经是另一项实验。

下面几个术语有助于区分不同层次：

- **可重复性（repeatability）：**同一个团队使用相同代码和设置重新运行，获得实质等价的结果；
- **可复现性（reproducibility）：**另一位研究者使用记录下来的工件和流程，获得实质等价的结果；
- **可复制性（replicability）：**独立实现和独立研究仍支持同一科学主张。

在并行硬件或非确定性加速器上，逐位完全一致并不总是可行。因此，应预先声明所要求的等价标准：它可以是完全相同的模型哈希、容差内一致的预测、落入同一置信区间的指标，或者方法之间保持相同的定性排序。

#### **数据、代码、环境与模型版本化**

一次运行应标识完整输入元组：

$$
r=
\bigl(
v_{\text{data}},
v_{\text{code}},
v_{\text{environment}},
c_{\text{config}},
s_{\text{seed}},
h_{\text{hardware}}
\bigr).
$$

- **数据版本**包括源快照、查询文本、提取时间、排除规则、划分分配和哈希。
- **代码版本**标识 commit，并记录工作区是否包含未提交更改。
- **环境版本**捕获语言、库、系统软件包、容器镜像和相关驱动。
- **配置**保存会改变预处理、训练、评估和选择过程的每个参数。
- **随机种子**控制实现实际遵循的伪随机操作。
- 当数值内核、精度、并行顺序或设备行为影响结果时，**硬件与执行模式**同样重要。

随机种子本身不是可复现策略。它不会冻结输入行顺序、依赖库行为、非确定性内核、隐藏全局状态或远程更新的数据集。很多情况下还需要多个独立种子来估计变异，而不是用单一种子掩盖它。

<div class="diagram-scroll">

![实验血缘把版本化输入连接到运行、工件与决策。](assets/experiment-lineage.svg){fig-alt="版本化数据、代码、环境和配置进入一次运行，再产生工件和决策记录。"}

</div>

<details>
<summary><strong>Python：使用内容哈希创建确定性实验清单</strong></summary>

```python
import hashlib
import json
import platform
import sys
import numpy as np
import sklearn


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


rng = np.random.default_rng(20)
X = rng.normal(size=(120, 5)).astype("float64")
y = (X[:, 0] + 0.4 * X[:, 1] > 0).astype("int8")

config = {
    "model": "logistic_regression",
    "regularization_C": 0.5,
    "split_seed": 20,
    "training_seed": 20,
    "feature_version": "customer_features_v3",
}

manifest = {
    "dataset_sha256": sha256_bytes(X.tobytes() + y.tobytes()),
    "config_sha256": sha256_bytes(
        json.dumps(config, sort_keys=True).encode("utf-8")
    ),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "platform": platform.platform(),
    # In a real run these values come from version control and the image registry.
    "code_commit": "9d3c02f",
    "container_digest": "sha256:example-image-digest",
}

run_id = sha256_bytes(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
)[:16]

print("run_id", run_id)
print(json.dumps(manifest, indent=2, sort_keys=True))
```

</details>

#### **随机种子、配置与工件管理**

实验追踪会把**元数据**与**工件**分开：

- 元数据是参数、指标、标签、时间戳、状态、数据集标识和父运行等体积较小、便于检索的信息；
- 工件是模型权重、预测、图表、序列化预处理器、环境锁文件和报告等较大文件。

这样，追踪数据库可以回答“哪些运行使用特征版本 3 且召回率高于 0.8”，工件存储则保存实际模型与预测文件。模型注册表进一步增加命名版本、血缘、审核状态、别名和部署引用。它本身不应被误当成批准系统；发布标准和负责任的签字仍需显式记录。

<div class="diagram-scroll wide-diagram">

![MLflow 追踪可以从本地文件演化为共享追踪服务器、元数据数据库与工件存储。](assets/mlflow-tracking-architecture-official.png){fig-alt="MLflow 官方图比较纯本地追踪、使用不同本地存储的追踪，以及包含服务器、数据库和云端工件存储的团队远程追踪。"}

</div>

*图片来源：[MLflow Architecture Overview](https://mlflow.org/docs/latest/self-hosting/architecture/overview/)，来自采用 [Apache-2.0 许可的 MLflow 项目](https://github.com/mlflow/mlflow/blob/master/LICENSE.txt)。*

每个记录的指标都必须带有上下文。若没有数据集与划分、总体、阈值、模型版本、指标实现和不确定性，`accuracy=0.91` 无法被正确解释。一份稳健的运行记录通常保存：

| 工件 | 作用 |
|---|---|
| 划分分配或稳定行标识 | 验证所有候选使用相同评估总体 |
| 原始预测与标签 | 允许重新计算指标、阈值、切片与不确定性 |
| 已拟合预处理对象 | 防止服务时使用不兼容变换 |
| 配置与环境锁文件 | 重建训练过程 |
| 训练曲线与资源日志 | 诊断收敛情况与计算成本 |
| 模型签名与 schema | 验证服务输入与输出 |
| 决策记录 | 解释某次运行为何晋升、被拒绝或被替代 |

<details>
<summary><strong>Python：实现包含不可变运行记录的最小实验追踪器</strong></summary>

```python
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
import hashlib
import json


@dataclass(frozen=True)
class RunRecord:
    run_id: str
    created_at: str
    parameters: dict
    metrics: dict
    artifacts: dict
    tags: dict = field(default_factory=dict)


def create_run(parameters, metrics, artifacts, tags=None):
    payload = {
        "parameters": parameters,
        "metrics": metrics,
        "artifacts": artifacts,
        "tags": tags or {},
    }
    run_id = hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    return RunRecord(
        run_id=run_id,
        created_at=datetime.now(timezone.utc).isoformat(),
        parameters=dict(parameters),
        metrics=dict(metrics),
        artifacts=dict(artifacts),
        tags=dict(tags or {}),
    )


registry = {}
record = create_run(
    parameters={"model": "logistic", "C": 0.5, "seed": 20},
    metrics={"validation_log_loss": 0.384, "validation_recall_at_500": 0.812},
    artifacts={
        "model": "artifacts/model.pkl",
        "predictions": "artifacts/validation_predictions.parquet",
        "environment": "artifacts/requirements.lock",
    },
    tags={"dataset": "events@2026-04-01", "code_commit": "9d3c02f"},
)
registry[record.run_id] = record

print(json.dumps(asdict(registry[record.run_id]), indent=2, sort_keys=True))
```

</details>

真实平台还会加入事务、身份认证、远程存储、血缘查询和生命周期策略。这个最小示例展示了概念契约：参数、指标、工件和来源共同构成一条不可变运行记录。会被下一次实验覆盖的可变电子表格行无法提供同等审计轨迹。

**对比总结。** 版本控制回答发生了什么变化；环境捕获回答实际执行了什么；实验追踪回答运行中发生了什么；工件存储保留证据；模型注册表标识已发布版本。这些工具单独使用都不能保证科学有效性，但组合起来能显著降低发现无效或不可复现结论的难度。


### **部署与服务**

部署把经过审核的实验工件转化为版本化服务或批处理任务，使其能够接收合法输入、产生定义明确的输出、满足运行目标并可以撤回。部署并不等同于复制一个序列化估计器。可部署单元通常包含：

- 特征 schema、顺序、类型、词表和缺失值策略；
- 预处理与后处理代码；
- 模型参数和推理实现；
- 输入与输出签名；
- 依赖与运行时版本；
- 健康检查、遥测和版本标识；
- 资源请求和并发限制；
- 降级与回滚说明。

发布前，软件包应通过变换函数的**单元测试**、schema 的**契约测试**、完整请求路径的**集成测试**、历史请求的**重放测试**、延迟与饱和度的**负载测试**，以及不变量与关键切片的**行为测试**。离线准确率无法发现缺失特征、不兼容的类别编码器、线程安全错误或过载的下游存储。

#### **批量、在线与边缘推理**

服务模式取决于决策截止时间和决策发生的位置。

<div class="diagram-scroll">

![批量、在线与边缘服务在新鲜度、延迟与资源约束之间取舍。](assets/serving-modes.svg){fig-alt="三个面板比较批量推理、低延迟在线推理与资源受限的边缘推理。"}

</div>

| 模式 | 最适合的场景 | 主要优点 | 主要风险 |
|---|---|---|---|
| 批量 | 分数按计划时间被消费 | 吞吐量高、向量化高效、容易重放 | 预测陈旧与大型任务整体失败 |
| 在线 | 行动需要等待请求时分数 | 上下文新鲜、即时决策 | 尾延迟、可用性、并发和在线特征故障 |
| 边缘 | 数据或行动必须留在设备或本地站点 | 隐私边界、离线运行、较少依赖网络 | 内存与能耗紧张、硬件多样、设备群更新缓慢 |

静态训练可以搭配动态在线推理，动态训练也可以搭配批量推理。**训练频率**和**服务频率**是相互独立的设计选择。如果底层关系每月才变化一次，每小时重训练只会浪费资源；而每年训练一次的模型仍可能需要在请求到达时以毫秒级速度返回预测。

#### **延迟、吞吐量、内存与成本**

端到端延迟由多个相互依赖的阶段组成：

$$
T_{\text{request}}
=T_{\text{network}}
+T_{\text{feature}}
+T_{\text{queue}}
+T_{\text{model}}
+T_{\text{post}}
+T_{\text{downstream}}.
$$

总和的 p99 通常不等于各组件 p99 之和，因为不同阶段可能相关，并且每个阶段进入尾部的并不是同一批请求。必须在真实并发条件下测量完整路径。当用户可见的超时由尾部请求造成时，平均延迟尤其容易误导。

吞吐量 $\lambda$ 表示单位时间内处理的请求数。对于稳定队列，若请求在系统中的平均时间为 $W$，Little 定律给出平均在途请求数：

$$
L=\lambda W.
$$

当到达速率超过服务能力时，排队延迟会急剧增长。批处理能提高硬件利用率，却可能增加等待时间。缓存减少计算，但必须定义缓存键、过期策略、失效逻辑，并分析结果陈旧带来的影响。

内存消耗包括模型参数、运行时开销、特征缓冲区、请求批次、缓存和每个 worker 复制的状态。成本应归一化到有意义的单位，例如每千次预测、每个接受案例或每个正确决策的成本，而不是只报告缺乏工作量背景的月度总额。

<details>
<summary><strong>Python：分解延迟并检验端到端服务目标</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n_requests = 100_000

# Shared load makes stages correlated: busy requests wait longer for both
# feature retrieval and model execution.
load = rng.lognormal(mean=-2.0, sigma=0.7, size=n_requests)
network = rng.gamma(2.0, 1.2, n_requests)
feature = rng.gamma(2.0, 2.0, n_requests) + 18.0 * load
queue = rng.exponential(1.0 + 12.0 * load)
model = rng.gamma(3.0, 1.4, n_requests) + 10.0 * load
post = rng.gamma(2.0, 0.5, n_requests)
end_to_end = network + feature + queue + model + post


def percentiles(values):
    return {
        "p50": round(float(np.quantile(values, 0.50)), 2),
        "p95": round(float(np.quantile(values, 0.95)), 2),
        "p99": round(float(np.quantile(values, 0.99)), 2),
    }


for name, values in {
    "network": network,
    "feature": feature,
    "queue": queue,
    "model": model,
    "post": post,
    "end_to_end": end_to_end,
}.items():
    print(name, percentiles(values))

latency_slo_ms = 80.0
print("p99_slo_pass", np.quantile(end_to_end, 0.99) <= latency_slo_ms)
print(
    "incorrect_sum_of_component_p99",
    round(sum(np.quantile(v, 0.99) for v in [network, feature, queue, model, post]), 2),
)
```

</details>

#### **打包、安全发布与回滚**

发布应当逐步增加暴露范围，同时保留已知可靠路径。

<div class="diagram-scroll">

![安全部署依次经过离线、影子、金丝雀、扩量与正式运行关卡。](assets/safe-rollout-stages.svg){fig-alt="五个部署阶段由证据关卡连接，失败时通过回滚循环返回已知良好版本。"}

</div>

- **影子部署（shadow deployment）**把生产请求副本发送给候选模型，但不让输出影响用户。它能在真实流量上发现 schema、延迟、数值和预测分布差异。
- **金丝雀部署（canary deployment）**把一小部分随机流量发送给候选模型，在有限暴露下测试运行与结果护栏。
- **A/B 测试**把合格单位随机分配给不同策略，以估计对产品或决策结果的因果影响。随机化单位必须避免干扰与同一用户重复进入不同组。
- **蓝绿部署（blue-green deployment）**同时维护两套完整环境并在二者之间切换流量，以更高基础设施成本换取快速回滚。

在观察金丝雀结果前就应声明晋升标准。回滚触发器可以组合错误率、p99 延迟、预测覆盖率和高严重度切片指标。降级方案可以是旧模型、确定性规则、缓存分数、拒绝预测或人工复核。当下游应用必须获得响应时，“关闭服务”并不是充分的降级方案。

<details>
<summary><strong>Python：根据运行护栏评估金丝雀版本</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n_control = 60_000
n_canary = 6_000

# Simulated request telemetry. The canary improves outcome utility slightly
# but has a heavier latency tail and a higher error probability.
control_latency = rng.lognormal(mean=3.35, sigma=0.34, size=n_control)
canary_latency = rng.lognormal(mean=3.45, sigma=0.43, size=n_canary)
control_errors = rng.binomial(1, 0.0012, n_control)
canary_errors = rng.binomial(1, 0.0030, n_canary)
control_utility = rng.normal(1.00, 0.50, n_control)
canary_utility = rng.normal(1.06, 0.50, n_canary)

report = {
    "control_p99_ms": float(np.quantile(control_latency, 0.99)),
    "canary_p99_ms": float(np.quantile(canary_latency, 0.99)),
    "control_error_rate": float(control_errors.mean()),
    "canary_error_rate": float(canary_errors.mean()),
    "utility_lift": float(canary_utility.mean() - control_utility.mean()),
}

guardrails = {
    "p99_latency_under_80ms": report["canary_p99_ms"] <= 80.0,
    "error_rate_under_0.002": report["canary_error_rate"] <= 0.002,
    "positive_utility_lift": report["utility_lift"] > 0,
}

print({key: round(value, 5) for key, value in report.items()})
print("guardrails", guardrails)
print("decision", "PROMOTE" if all(guardrails.values()) else "ROLL_BACK")
```

</details>

即使候选模型的平均效用看起来更高，它仍会因为预先定义的错误率护栏失败而回滚。在真实数据上，还必须更谨慎地处理不确定性、序贯监控、最小样本量、季节性和多指标问题。核心原则不变：暴露过程必须可逆，晋升必须由证据驱动。

**对比总结。** 批量服务优化吞吐量，在线服务优化请求时新鲜度，边缘服务优化本地性。影子部署在无用户影响下检验兼容性，金丝雀限制运行风险，A/B 测试估计策略效果。打包定义实际执行内容，回滚定义系统如何安全失败。


### **监控与维护**

监控是持续收集证据、检测可行动偏差、诊断可能原因并触发有明确负责人的响应过程。一组仪表板只能提供可观测性，还不是完整运行控制。每条告警都需要：

1. 明确的信号与聚合窗口；
2. 具有已知误报行为的阈值或统计规则；
3. 负责人和严重等级；
4. 诊断手册；
5. 调查、优雅降级、回滚、重训练或停止等行动；
6. 事故及其解决过程的记录。

监控必须覆盖从基础设施到最终结果的完整链路。

<div class="diagram-scroll">

![监控横跨系统、数据、模型与决策层。](assets/monitoring-layers.svg){fig-alt="四个面板分别列出系统基础设施、数据、模型行为和下游决策的可观测信号。"}

</div>

#### **数据质量、漂移与性能衰减**

**数据质量监控**检查当前输入是否满足契约。典型信号包括 schema 变化、缺失、重复、非法类别、范围违规、新鲜度、数据量、实体覆盖率和特征计算失败。

**分布监控**比较当前总体 $Q(X)$ 与参考总体 $P(X)$。它可以检测协变量变化，却不能单独判断模型是否已经变差。性能取决于 $X$、$Y$ 和决策策略之间的联合关系。某个特征可能发生漂移而预测依然有效；反过来，边际特征分布看似稳定时，准确率也可能崩溃。

常见单变量漂移统计量包括：

- 连续经验分布的 Kolmogorov-Smirnov 距离；
- 离散分布的 Jensen-Shannon 散度；
- 数值几何关系重要时使用 Wasserstein 距离；
- 使用固定参考分箱的总体稳定性指数（PSI）；
- 缺失率、类别频率、分位数或越界率变化。

若参考分箱比例为 $p_b$，当前比例为 $q_b$，则 PSI 为

$$
\operatorname{PSI}(P,Q)
=
\sum_{b=1}^{B}
(q_b-p_b)\log\frac{q_b}{p_b}.
$$

分箱为空时需要平滑处理。PSI 没有通用运行阈值：它会随分箱方式、样本量、平滑方法和特征分布而变化。应使用历史正常时期与已知事故校准阈值，并结合特征重要性、模型行为和切片诊断来解释。

<details>
<summary><strong>Python：使用固定参考分箱计算 PSI 并检查其敏感性</strong></summary>

```python
import numpy as np


def population_stability_index(reference, current, bins=10, epsilon=1e-6):
    # Quantile edges are learned only from the reference period and then frozen.
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    edges = np.unique(edges)

    reference_counts, _ = np.histogram(reference, bins=edges)
    current_counts, _ = np.histogram(current, bins=edges)

    p = reference_counts / reference_counts.sum()
    q = current_counts / current_counts.sum()
    p = np.clip(p, epsilon, None)
    q = np.clip(q, epsilon, None)
    contributions = (q - p) * np.log(q / p)
    return float(contributions.sum()), edges, contributions


rng = np.random.default_rng(20)
reference = rng.normal(loc=0.0, scale=1.0, size=20_000)
benign_current = rng.normal(loc=0.05, scale=1.0, size=20_000)
shifted_current = rng.normal(loc=0.70, scale=1.25, size=20_000)

for name, sample in {
    "benign": benign_current,
    "shifted": shifted_current,
}.items():
    psi, edges, contributions = population_stability_index(reference, sample)
    print(name, "psi", round(psi, 4), "largest_bin_contribution", round(contributions.max(), 4))
```

</details>

既要监控原始输入，也要监控模型实际使用的特征。健康的原始数据源仍可能进入损坏的变换；特征分布保持稳定时，特征与结果之间的映射也可能改变。预测监控应包含分数分布、预测类别比例、置信度、校准度、拒绝预测、覆盖率和关键切片。条件允许时，应在同一批记录流量上比较不同模型版本。

#### **延迟标签与生产性能**

很多结果在预测后很久才到达，例如拒付、贷款违约、订阅续费、康复情况、疾病进展或长期参与度。因此，生产评估存在两个时钟。

<div class="diagram-scroll">

![延迟标签评估把成熟结果连接回历史预测记录。](assets/delayed-label-clocks.svg){fig-alt="四个相连阶段依次表示预测时刻、结果延迟成熟、评估和运行行动。"}

</div>

每条预测日志都应包含稳定样本或实体键、事件时间、模型版本、特征版本、分数、决策、资格状态和相关切片属性。标签成熟后，评估应使用固定 cohort：

$$
\mathcal{C}_t
=
\left\{
i:
t_i\le t-h
\;\land\;
\text{outcome window for }i\text{ is complete}
\right\},
$$

其中 $h$ 是标签时间范围。把已经成熟的旧案例与尚未成熟的新案例混合会造成截尾偏差。报告必须包含**标签覆盖率**和标签缺失原因。如果行动会影响可观测性，例如只复核高风险案例，那么有标签子集带有选择性，普通准确率估计可能有偏。

<details>
<summary><strong>Python：连接延迟结果，并且只评估成熟 cohort</strong></summary>

```python
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score

predictions = pd.DataFrame(
    {
        "case_id": ["a", "b", "c", "d", "e", "f"],
        "prediction_time": pd.to_datetime(
            ["2026-01-01", "2026-01-05", "2026-01-20", "2026-02-01", "2026-02-15", "2026-03-10"],
            utc=True,
        ),
        "model_version": ["v7", "v7", "v7", "v8", "v8", "v8"],
        "score": [0.80, 0.25, 0.60, 0.35, 0.72, 0.55],
    }
)

outcomes = pd.DataFrame(
    {
        "case_id": ["a", "b", "c", "d", "e"],
        "outcome_time": pd.to_datetime(
            ["2026-01-12", "2026-01-25", "2026-02-11", "2026-02-20", "2026-03-12"],
            utc=True,
        ),
        "label": [1, 0, 1, 0, 1],
    }
)

as_of = pd.Timestamp("2026-04-01", tz="UTC")
label_horizon = pd.Timedelta(days=30)
joined = predictions.merge(outcomes, on="case_id", how="left")
joined["mature"] = joined["prediction_time"] + label_horizon <= as_of

mature = joined[joined["mature"]]
labeled = mature.dropna(subset=["label"]).copy()
coverage = len(labeled) / len(mature)

print("mature_cases", len(mature), "label_coverage", round(coverage, 3))
print("auc", round(roc_auc_score(labeled["label"], labeled["score"]), 3))
print("log_loss", round(log_loss(labeled["label"], labeled["score"]), 3))
print(labeled.groupby("model_version").size().rename("evaluated_cases"))
```

</details>

标签成熟前，输入和分数漂移是很有用的诊断**代理信号**，但不能证明预测性能已经变化。快速代理告警可以触发调查；晋升、回滚或重训练则应使用当时能够获得的最强证据，并明确承认标签延迟。

#### **服务级目标与事故响应**

**服务级指标（SLI）**是可测量量，例如 80 ms 内成功返回预测的响应比例。**服务级目标（SLO）**是这个指标在给定窗口内的目标，例如 30 天内达到 99.9%。允许失败的比例构成**错误预算**：

$$
\text{error budget}
=
N\left(1-\text{SLO target}\right)
$$

其中 $N$ 是合格请求数。错误预算把模糊的可靠性愿望转化为明确取舍。预算被快速消耗时，可以暂停高风险发布并优先处理可靠性工作。ML 服务通常需要多个 SLO：可用性、延迟、有效特征覆盖率、预测覆盖率和新鲜度。

事故应按层级进行分诊：

1. **系统：**超时、资源饱和、依赖故障、部署错误。
2. **数据：**schema、数据量、新鲜度、非法值、源系统中断。
3. **模型：**分数坍缩、校准漂移、不稳定切片、错误版本。
4. **决策：**工作量超载、策略变化、不利反馈、意外伤害。

响应过程必须保留证据。记录时间线、受影响版本与 cohort、自动行动、人工决策、用户影响、根因和预防措施。删除失败工件或改写原始实验会破坏事故最重要的学习价值。

<details>
<summary><strong>Python：计算错误预算消耗速度并映射为运行行动</strong></summary>

```python
requests_30d = 12_000_000
slo_target = 0.999
allowed_failures = requests_30d * (1.0 - slo_target)

# Four hours into a 30-day window, a dependency incident has caused failures.
elapsed_hours = 4
window_hours = 30 * 24
observed_failures = 1850

expected_budget_by_now = allowed_failures * elapsed_hours / window_hours
burn_rate = observed_failures / expected_budget_by_now
remaining_budget = allowed_failures - observed_failures

if burn_rate >= 14.4:
    action = "PAGE_AND_ROLL_BACK"
elif burn_rate >= 6.0:
    action = "PAGE_AND_FREEZE_RELEASES"
elif burn_rate >= 2.0:
    action = "INVESTIGATE"
else:
    action = "CONTINUE_MONITORING"

print("allowed_failures_30d", round(allowed_failures))
print("observed_failures", observed_failures)
print("remaining_budget", round(remaining_budget))
print("burn_rate", round(burn_rate, 2))
print("action", action)
```

</details>

**对比总结。** 契约检查发现非法输入；漂移指标发现总体变化；预测遥测发现模型行为变化；成熟标签估计预测质量；决策结果估计真实价值。SLO 管理服务可靠性，事故响应则把故障转化为有明确负责人的纠正行动。


### **反馈循环与重训练策略**

预测一旦影响决策，模型就成为数据生成过程的一部分。推荐系统改变曝光，欺诈模型改变攻击者行为，筛查模型改变哪些案例会接受确诊检测，风险分数改变哪些用户会接受干预。因此，下一份数据集会以历史策略为条件。

<div class="diagram-scroll">

![模型分数会改变决策、观测结果和未来训练数据。](assets/feedback-loop-dynamics.svg){fig-alt="循环连接模型分数、决策、观测数据和重训练，表示预测会改变未来样本。"}

</div>

需要区分多种反馈机制：

- **选择性标签：**只有接受某项行动的单位才能观察到结果，例如只有获批贷款才能观察还款情况；
- **曝光偏差：**排序系统主要观察被展示项目的互动；
- **自我实现预测：**干预让预测结果更可能发生；
- **自我否定预测：**干预阻止预测结果，使模型看起来像是预测错误；
- **行为适应：**用户、市场或攻击者作出策略性响应；
- **总体构成变化：**策略改变谁会进入或留在系统中；
- **测量反馈：**模型改变标签或特征的记录方式。

假设历史策略在分数 $S$ 超过阈值 $\tau$ 时批准申请，则还款标签 $Y$ 主要在

$$
A=\mathbb{1}[S\ge\tau]=1
$$

时可见。在 $P(Y\mid X,A=1)$ 上训练，通常无法恢复被拒申请的 $P(Y\mid X)$。因为决策依赖与风险有关的特征，所以缺失标签并非随机缺失。直接把未观测结果当作负例会造成更强偏差。

缓解方法取决于具体机制：

- 在伦理和运行允许时保留随机探索样本；
- 使用随机或准实验数据估计策略效果；
- 收集决策路径之外的独立审计或延迟标签；
- 只有在假设合理时，才对观测倾向建模并加权；
- 记录资格、曝光、行动、人工覆盖和观测状态；
- 在不依赖当前策略或由外部标注的 cohort 上评估；
- 对升级案例引入人类，但同时测量人类分歧和自动化偏见。

<details>
<summary><strong>Python：展示历史策略产生的选择性标签偏差</strong></summary>

```python
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(20)
def generate_population(size):
    income = rng.normal(size=size)
    risk = rng.normal(size=size)
    # A nonlinear interaction is especially important in the low-income region
    # that the historical policy rarely labels.
    logit = (
        -0.3 + 0.9 * income - 1.2 * risk
        + 2.2 * (income < -0.6) * risk
    )
    probability = 1 / (1 + np.exp(-logit))
    label = rng.binomial(1, probability)
    return np.column_stack([income, risk]), label


X_train, y_train = generate_population(60_000)
X_test, y_test = generate_population(30_000)

# Historical approval depends on a noisy score related to the outcome. Labels
# from rejected training cases are hidden, but the complete test population is
# available in this simulation for auditing.
historical_score = (
    0.9 * X_train[:, 0] - 0.9 * X_train[:, 1]
    + rng.normal(0, 0.45, len(X_train))
)
approved = historical_score > 0.2

observed_model = HistGradientBoostingClassifier(
    max_iter=120, max_leaf_nodes=15, random_state=20
).fit(X_train[approved], y_train[approved])
oracle_model = HistGradientBoostingClassifier(
    max_iter=120, max_leaf_nodes=15, random_state=20
).fit(X_train, y_train)

observed_auc_all = roc_auc_score(
    y_test, observed_model.predict_proba(X_test)[:, 1]
)
oracle_auc_all = roc_auc_score(
    y_test, oracle_model.predict_proba(X_test)[:, 1]
)

print("training_label_coverage", round(approved.mean(), 3))
print("approved_positive_rate", round(y_train[approved].mean(), 3))
print("rejected_positive_rate_hidden_from_training", round(y_train[~approved].mean(), 3))
print("observed_only_model_auc_on_full_population", round(observed_auc_all, 4))
print("oracle_model_auc_on_full_population", round(oracle_auc_all, 4))
```

</details>

代码之所以能查看被拒申请的结果，是因为它来自模拟。真实系统往往无法获得这些结果，这正是为什么已批准案例上的高验证性能不能证明模型在全部申请者上同样有效。

#### **重训练触发条件**

重训练是对已部署决策系统的受控变更，而不是例行清理工作。常见策略包括：

- **定期触发：**每天、每周或每月重训练；
- **数据量触发：**积累足够多的新成熟样本后重训练；
- **漂移触发：**输入或分数持续且显著变化后重训练；
- **性能触发：**成熟标签显示具有统计可信度的性能下降后重训练；
- **事件触发：**产品、策略、传感器、词表或市场发生变化后重训练；
- **人工批准：**负责人审核证据并启动运行。

每种触发器都有局限。定期重训练可能用噪声更大的模型替换健康模型；漂移不能证明新模型会更好；标签延迟时，性能告警来得很晚；事件触发则依赖组织内部沟通。

稳健策略会把**触发**、**候选训练**、**验证**与**晋升**分离：

$$
\text{signal}
\rightarrow
\text{train candidate}
\rightarrow
\text{compare with incumbent}
\rightarrow
\text{release gates}
\rightarrow
\text{monitor}.
$$

重训练从不意味着自动晋升。现有模型仍是基线和回退方案。使用滞回、持续窗口和冷却时间防止系统来回震荡：

- 只有下降幅度连续 $k$ 个窗口超过 $\tau_{\text{high}}$ 才触发；
- 只有下降幅度回到 $\tau_{\text{low}}<\tau_{\text{high}}$ 以下才关闭事故；
- 发布或训练失败后等待一段冷却时间；
- 要求成熟标签数量足够，且不确定性范围足够窄。

<details>
<summary><strong>Python：实现带持续要求与冷却期的有状态重训练触发器</strong></summary>

```python
from dataclasses import dataclass


@dataclass
class RetrainingPolicy:
    high_threshold: float = 0.045
    low_threshold: float = 0.025
    persistence_windows: int = 3
    cooldown_windows: int = 4
    consecutive_high: int = 0
    cooldown_remaining: int = 0
    incident_open: bool = False

    def update(self, performance_drop: float, mature_labels: int) -> str:
        if self.cooldown_remaining > 0:
            self.cooldown_remaining -= 1
            return "COOLDOWN"

        if mature_labels < 1000:
            return "WAIT_FOR_LABELS"

        if performance_drop >= self.high_threshold:
            self.consecutive_high += 1
        else:
            self.consecutive_high = 0

        if self.incident_open and performance_drop <= self.low_threshold:
            self.incident_open = False
            return "RECOVERED"

        if self.consecutive_high >= self.persistence_windows:
            self.incident_open = True
            self.consecutive_high = 0
            self.cooldown_remaining = self.cooldown_windows
            return "TRAIN_CANDIDATE"

        return "MONITOR"


policy = RetrainingPolicy()
windows = [
    (0.018, 1400), (0.052, 1500), (0.049, 1550), (0.051, 1600),
    (0.060, 1700), (0.040, 1800), (0.020, 1900), (0.055, 2000),
]

for index, (drop, labels) in enumerate(windows, start=1):
    print(index, drop, policy.update(drop, labels))
```

</details>

触发 `TRAIN_CANDIDATE` 后，流水线应快照合格数据、产生版本化运行、在不变的发布标准上与现有模型比较，然后拒绝候选或安全部署。触发器只能证明值得展开调查，不能证明最新数据或最新架构一定能解决问题。

**对比总结。** 普通漂移假设世界独立于模型而变化；反馈循环则承认策略会改变观测与行为。重训练会重新启动实验周期，但晋升仍然需要基线比较、不确定性、约束和可逆部署。


### **可扩展性与高效学习**

可扩展性是指当数据量、特征维度、模型规模、请求速率或团队使用量增长时，训练与服务过程仍然可行。它不等同于分布式训练。首先要判断真正受限的资源：

- CPU 或加速器计算能力；
- 主机或设备内存；
- 存储容量与读取带宽；
- 网络通信；
- 特征查询延迟；
- 请求并发量；
- 标注或人工复核容量；
- 实验周转时间；
- 能源或资金预算。

优化前必须先测量。性能分析可能发现训练任务主要耗时在特征连接，而 GPU 利用率仍然很低；也可能发现在线模型本身很快，但远程特征查询主导 p99 延迟。扩展错误组件只会增加成本，不会提高吞吐量。

<div class="diagram-scroll">

![资源感知的模型选择应在效用与成本前沿上选择可行点。](assets/resource-aware-frontier.svg){fig-alt="曲线展示效用随计算、延迟与内存成本增加而边际递减，并标出效用要求和资源约束。"}

</div>

#### **并行、近似与资源感知建模**

常见扩展策略作用于不同层次：

- **数据并行：**worker 处理不同样本并聚合梯度或统计量，通信和同步可能成为瓶颈。
- **模型并行：**当单个设备放不下模型时，把参数或层分散到多个设备。
- **流水线并行：**不同模型阶段并发处理不同 micro-batch，在利用率与调度复杂度之间取舍。
- **向量化与批处理：**使用高效内核并摊薄开销，不改变模型本身。
- **采样与 sketch：**以可控近似减少行、负样本、类别或充分统计量。
- **稀疏表示：**避免存储或乘以零值。
- **量化：**用更少位数表示权重或激活，需要重新验证精度与硬件行为。
- **剪枝：**移除权重、分支、特征或专家；只有运行时真正利用稀疏性时，非结构化稀疏才会加速。
- **蒸馏：**训练较小学生模型逼近较大教师模型。
- **缓存与预计算：**以新鲜度和存储空间换取更低请求时计算量。

对候选模型 $m$，定义向量

$$
z_m=
\bigl(
\text{utility}_m,
-\text{latency}_m,
-\text{memory}_m,
-\text{cost}_m
\bigr).
$$

如果候选 $a$ 在每个维度上都不差于 $b$，且至少一个维度严格更好，就称 $a$ **支配** $b$。被支配模型不需要主观权重：另一个候选在所有已记录标准上都更好。剩余 Pareto 前沿会把真正需要决策者取舍的方案暴露出来。

<details>
<summary><strong>Python：移除被支配模型并执行硬资源约束</strong></summary>

```python
candidates = [
    {"name": "linear", "utility": 0.742, "p99_ms": 6, "memory_mb": 8, "cost": 0.02},
    {"name": "small_tree", "utility": 0.781, "p99_ms": 11, "memory_mb": 28, "cost": 0.05},
    {"name": "boosted", "utility": 0.824, "p99_ms": 38, "memory_mb": 180, "cost": 0.35},
    {"name": "ensemble", "utility": 0.831, "p99_ms": 96, "memory_mb": 720, "cost": 1.80},
    {"name": "compressed", "utility": 0.816, "p99_ms": 22, "memory_mb": 95, "cost": 0.18},
    {"name": "legacy", "utility": 0.760, "p99_ms": 45, "memory_mb": 240, "cost": 0.60},
]


def dominates(a, b):
    no_worse = (
        a["utility"] >= b["utility"]
        and a["p99_ms"] <= b["p99_ms"]
        and a["memory_mb"] <= b["memory_mb"]
        and a["cost"] <= b["cost"]
    )
    strictly_better = (
        a["utility"] > b["utility"]
        or a["p99_ms"] < b["p99_ms"]
        or a["memory_mb"] < b["memory_mb"]
        or a["cost"] < b["cost"]
    )
    return no_worse and strictly_better


frontier = [
    candidate
    for candidate in candidates
    if not any(dominates(other, candidate) for other in candidates if other is not candidate)
]
feasible = [
    candidate
    for candidate in frontier
    if candidate["p99_ms"] <= 50
    and candidate["memory_mb"] <= 256
    and candidate["cost"] <= 0.50
]

print("pareto_frontier", [candidate["name"] for candidate in frontier])
print("feasible_frontier", [candidate["name"] for candidate in feasible])
print("best_feasible_utility", max(feasible, key=lambda row: row["utility"]))
```

</details>

`legacy` 模型被移除，因为另一个候选在所有记录维度上都更好。集成模型仍属于非支配解，但违反硬服务契约。只有当测得的质量-资源权衡优于直接训练小模型时，模型压缩才真正有价值。

### **AutoML 与自动模型选择**

自动机器学习会搜索一个预先定义的流水线空间。根据范围不同，搜索变量可以包括：

- 插补、编码、缩放、特征选择与重采样；
- 模型家族与超参数；
- 架构、数据增强、优化器与调度；
- 阈值、校准、集成与压缩选择；
- 资源分配与提前停止。

自动化不会替你选择科学问题、阻止泄漏、论证指标合理性、定义部署约束或建立外部有效性。它只会放大既有协议。如果在交叉验证前就拟合预处理，大规模搜索只会比小规模搜索更高效地优化泄漏。

常见搜索策略包括：

- **网格搜索：**系统性强，但维度增加时组合数呈指数增长；
- **随机搜索：**当只有少数维度真正重要时通常更强；
- **贝叶斯优化：**对配置与性能之间的关系建模，并选择信息量高的试验；
- **进化或种群方法：**对候选执行变异与重组；
- **多保真方法：**先分配小预算，再晋升有潜力的配置；
- **Bandit 调度器：**根据中间证据提前停止较弱试验。

在 successive halving 中，先以资源 $r_0$ 评估 $n_0$ 个配置。若缩减因子 $\eta>1$，每轮大约保留 $1/\eta$ 个候选，并把资源乘以 $\eta$：

$$
n_k\approx\frac{n_0}{\eta^k},
\qquad
r_k=r_0\eta^k.
$$

资源可以是样本数、迭代次数、树数量、epoch 或数据分辨率。该方法假设低预算性能对高预算性能有信息价值，因此可能错误淘汰起步慢但最终很强的配置。必须检查配置排序和学习曲线。

<details>
<summary><strong>Python：在无泄漏协议下实现正则化参数的 successive halving</strong></summary>

```python
import math
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=7000,
    n_features=30,
    n_informative=10,
    n_redundant=5,
    weights=[0.72, 0.28],
    random_state=20,
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=1500, stratify=y, random_state=20
)

rng = np.random.default_rng(20)
order = rng.permutation(len(X_train))
X_train, y_train = X_train[order], y_train[order]

candidates = [{"C": value} for value in np.logspace(-3, 2, 9)]
resources = [600, 1800, len(X_train)]
reduction_factor = 3

for round_index, resource in enumerate(resources, start=1):
    scored = []
    for config in candidates:
        # Scaling is fitted inside each candidate pipeline on only the allocated
        # training subset; the validation set remains untouched.
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=config["C"], max_iter=2000),
        )
        model.fit(X_train[:resource], y_train[:resource])
        probability = model.predict_proba(X_validation)[:, 1]
        scored.append((log_loss(y_validation, probability), config))

    scored.sort(key=lambda item: item[0])
    keep = 1 if round_index == len(resources) else max(
        1, math.ceil(len(scored) / reduction_factor)
    )
    candidates = [config for _, config in scored[:keep]]
    print(
        "round", round_index,
        "resource", resource,
        "best_log_loss", round(scored[0][0], 4),
        "survivors", [round(item["C"], 5) for item in candidates],
    )
```

</details>

搜索选出流水线后，才应在最终未触碰测试集上评估。如果使用 AutoML 的目的不只是选模型，还要报告无偏性能估计，就需要把整个搜索嵌套进外层验证循环，或另设测试集。搜索预算、失败试验和所有尝试过的配置都是实验记录的一部分。

**对比总结。** 系统扩展解决真实瓶颈；模型压缩改变质量-资源前沿；分布式执行增加容量；AutoML 在搜索空间中分配实验预算。它们都不能替代有效的问题契约和无泄漏评估。


### **阅读与复现研究论文**

阅读论文的目标是重建主张及其证据，而不是收集架构名称。一篇论文可能同时包含理论主张、经验主张、工程主张和定性观察，每类主张需要不同验证方法。首先应把核心陈述改写成可检验形式：

> 在总体 $P$、数据集与划分协议 $D$、资源预算 $B$、指标 $M$ 和比较对象集合 $\mathcal{C}$ 下，方法 $A$ 以不确定性 $U$ 改善了数量 $\Delta$。

如果某个要素缺失，应把它标记为假设，而不是无声补齐。一个数据集上的一次随机划分结果不能自动推广为整个领域的结论；使用更大预训练语料或更高搜索预算得到的提升，也不能被直接解释为纯架构提升。

<div class="diagram-scroll">

![可信研究把精确主张连接到协议、证据、挑战测试与工件。](assets/research-evidence-chain.svg){fig-alt="五个阶段依次把主张连接到实验协议、证据、挑战测试和可执行工件。"}

</div>

#### **恢复假设与实验协议**

结构化初读可以采用五轮：

1. **主张轮：**识别问题、所提贡献、比较对象和适用范围。
2. **方法轮：**推导数学目标、数据流、推理规则和复杂度。
3. **协议轮：**恢复数据集、划分单位、预处理、超参数搜索、随机种子、计算预算、选择规则与指标。
4. **证据轮：**检查基线、不确定性、消融、敏感性、负结果和失败案例。
5. **工件轮：**检查代码、数据访问、环境、命令、检查点、许可，以及论文与实现之间的差异。

可以建立如下主张表：

| 主张 | 必需比较 | 关键控制 | 缺失时的威胁 |
|---|---|---|---|
| 预测方法更好 | 同等数据与预算下调优充分的强基线 | 相同划分、指标、预处理与搜索预算 | 提升可能来自协议优势 |
| 数据效率更高 | 随标注样本量变化的学习曲线 | 相同未标注数据与预训练 | 隐藏的监督优势 |
| 更鲁棒 | 指明偏移类型与严重程度 | 处于可比的干净性能区间 | 鲁棒性可能只源于较低干净容量 |
| 更快 | 端到端墙钟时间与硬件利用率 | 相同硬件、精度、batch 和质量目标 | 内核级加速可能不改善整体流程 |
| 更可解释 | 定义受众、解释目标与忠实度测试 | 预测质量可比 | 把合理性误当忠实度 |

重要细节经常藏在附录、配置文件、脚本、issue tracker 或数据加载器中。需要恢复：

- 精确的训练、验证和测试标识，以及实体是否跨划分；
- 数据排除与去重规则；
- checkpoint 选择和提前停止；
- 预处理在哪些行上拟合；
- 试验数量、随机种子和失败运行；
- 硬件、数值精度与运行时间；
- 开发过程中是否反复使用测试集；
- 表格展示最佳运行、均值、中位数还是集成结果。

缺失细节会按其改变结论的能力，相应降低结论可信度。

#### **从零实现与参考实现复现**

不同复现目标回答不同问题：

- **参考实现重跑：**执行作者代码并恢复报告中的行为；
- **净室重实现：**不复制作者代码，只根据论文实现方法；
- **组件验证：**在受控合成问题上验证数学或算法组件；
- **复制研究：**在新数据、环境、实现或团队上评估原主张；
- **扩展研究：**改变一个假设，研究结论何时成立或失效。

应从最小可执行案例开始。检查张量形状、损失项、mask、归一化、边界条件和小数据过拟合测试。对于概率方法，检查归一化，并与可精确求解的小案例比较；对于优化过程，检查梯度和学习曲线；对于数据流水线，手工审计几个样本从原始源到模型输入的全过程。

复现日志应区分：

1. **论文规范：**正文与公式写了什么。
2. **参考实现行为：**发布代码实际做了什么。
3. **你的实现：**为解决歧义所做的选择。
4. **观察到的差异：**数值或行为差异。
5. **调查过程：**假设、受控测试与证据。
6. **结论：**得到支持的范围、未解决不确定性和工件版本。

只匹配一个标题数字，弱于同时匹配学习曲线、消融排序、定性失败模式与资源画像。反过来，如果硬件、随机性或数据访问不同，数值不匹配也不自动推翻主张；应报告运行前确定的等价标准。

#### **消融、敏感性与失败分析**

消融实验询问某个组件在当前协议下是否必要；敏感性研究询问结论如何随合理参数、数据或环境选择变化；失败分析则询问方法在何处、为何失效。

良好的实验实践包括：

- 单因素消融与选定交互；
- 完整模型和消融版本使用相等调优预算；
- 在完全相同样本上成对评估；
- 优化具有随机性时运行多个训练种子；
- 置信区间围绕正确单位计算，例如用户而不是行；
- 检验大量假设时进行校正，或克制地解释结果；
- 学习曲线与资源-质量曲线；
- 独立于有利结果定义切片与定性错误分析；
- 报告负结果和不稳定配置。

对于配对预测，应重采样评估单位并重新计算**差值**，而不是比较两个互不相关的置信区间。如果同一用户或文档的多行相互依赖，应重采样群组。

<details>
<summary><strong>Python：估计消融增益的成对 bootstrap 区间</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n_groups = 600
rows_per_group = 5
group_id = np.repeat(np.arange(n_groups), rows_per_group)

latent = rng.normal(size=n_groups)
y = np.repeat((latent > 0).astype(int), rows_per_group)
y = np.where(rng.random(len(y)) < 0.12, 1 - y, y)

# Full and ablated model predictions are deliberately correlated because they
# are evaluated on the same rows.
base_signal = np.repeat(latent, rows_per_group)
full_score = 1 / (1 + np.exp(-(base_signal + rng.normal(0, 0.65, len(y)))))
ablated_score = 1 / (1 + np.exp(-(0.78 * base_signal + rng.normal(0, 0.78, len(y)))))


def binary_log_loss(labels, probability):
    probability = np.clip(probability, 1e-9, 1 - 1e-9)
    return -np.mean(
        labels * np.log(probability) + (1 - labels) * np.log(1 - probability)
    )


observed_gain = binary_log_loss(y, ablated_score) - binary_log_loss(y, full_score)
bootstrap_gains = []
for _ in range(2000):
    sampled_groups = rng.integers(0, n_groups, size=n_groups)
    sampled_rows = np.concatenate(
        [np.flatnonzero(group_id == group) for group in sampled_groups]
    )
    gain = (
        binary_log_loss(y[sampled_rows], ablated_score[sampled_rows])
        - binary_log_loss(y[sampled_rows], full_score[sampled_rows])
    )
    bootstrap_gains.append(gain)

lower, upper = np.quantile(bootstrap_gains, [0.025, 0.975])
print("full_model_log_loss_gain", round(observed_gain, 4))
print("group_paired_95_percent_interval", (round(lower, 4), round(upper, 4)))
```

</details>

该区间只涵盖当前协议下抽样群组总体带来的不确定性。除非在重采样设计中显式重跑，否则它不包含更换数据集、预处理、超参数搜索或训练种子产生的不确定性。

**对比总结。** 参考实现重跑验证工件；净室实现检验论文是否充分规定方法；复制研究检验主张能否在变化场景中存活。消融用于归因提升，敏感性用于描绘假设，失败分析用于定义有效性边界。


### **沟通机器学习结果**

沟通本身是证据流水线的一部分。如果一项结果无法连接到总体、协议、比较对象、不确定性估计和运行后果，它就还不足以支持决策。技术报告的目的不是陈列每项实验，而是让推理过程可以被审计。

#### **技术报告、视觉证据与局限性**

一份强报告可以按照审核者真正需要信息的顺序组织：

1. **决策与建议：**正在考虑什么选择，现有证据支持什么。
2. **范围：**预期总体、时间段、用途、排除项与非目标。
3. **数据与标签：**来源、单位、划分、成熟度、泄漏控制与局限。
4. **协议：**基线、候选、搜索预算、种子、选择规则、指标与计算资源。
5. **结果：**效应量、不确定性、切片、资源指标与失败案例。
6. **结论稳健性：**消融、敏感性、替代设定与负结果。
7. **运行计划：**打包、发布、监控、回退、负责人和退役条件。
8. **局限与开放问题：**研究没有证明什么。
9. **复现记录：**代码、数据、环境、命令与工件标识。

应同时报告**绝对值**和**差值**。相对提升可能夸大较小变化：

$$
\text{relative reduction}
=
\frac{e_{\text{baseline}}-e_{\text{candidate}}}
{e_{\text{baseline}}},
\qquad
\text{absolute reduction}
=
e_{\text{baseline}}-e_{\text{candidate}}.
$$

把错误率从 $2.0\%$ 降到 $1.5\%$，既可以说相对下降 25%，也可以说绝对下降 0.5 个百分点。二者都正确，但如果没有受影响决策数、错误成本、不确定性和切片表现，仍然不够完整。

结果表必须暴露比较契约：

| 模型 | 数据与预算 | 主要指标 | 95% 区间 | 最差关键切片 | p99 延迟 | 成本 | 决策 |
|---|---|---:|---:|---:|---:|---:|---|
| 现有模型 | 版本 A、预算 B | 数值 | 区间 | 数值 | 数值 | 数值 | 保留/晋升 |
| 候选模型 | 相同设定或有理由的差异 | 数值 | 区间 | 数值 | 数值 | 数值 | 保留/晋升 |

图像应回答一个明确问题：

- 学习曲线显示提升是否来自数据规模；
- 校准图显示概率是否可靠；
- 资源-质量前沿显示方案是否可行；
- 成对差值图显示一种方法在哪些样本上获胜；
- 切片矩阵显示故障是否集中；
- 时间线连接部署、数据变化与事故；
- 失败案例集揭示反复出现的定性模式。

应避免装饰性图表、扭曲效应的截断坐标轴、没有说明的平滑处理和只展示最佳种子。每张图都要说明总体、聚合方式、不确定性和改善方向。原始预测与绘图代码应保留在工件包中。

最有价值的局限性会精确说明哪项推断尚不被支持：

- “评估覆盖 2025 年澳大利亚的现有用户；它不能证明新用户或其他国家的性能。”
- “只有被复核案例存在标签，因此完整总体召回率不可识别。”
- “金丝雀持续七天，没有覆盖年度季节性。”
- “研究比较相同墙钟搜索预算，但没有比较相同能耗。”
- “置信区间涵盖测试样本，但不涵盖重训练种子变化。”

这些陈述能指导后续工作；“更多数据可能有帮助”这样的泛化措辞则不能。

<details>
<summary><strong>Python：构造一张拒绝不完整证据的决策表</strong></summary>

```python
import pandas as pd

rows = [
    {
        "model": "incumbent_v7",
        "dataset": "events@2026-04-01",
        "search_budget_gpu_h": 0,
        "primary_metric": 0.781,
        "ci_low": 0.768,
        "ci_high": 0.793,
        "worst_slice_recall": 0.744,
        "p99_latency_ms": 24,
        "cost_per_1000": 0.35,
        "decision": "retain",
    },
    {
        "model": "candidate_v8",
        "dataset": "events@2026-04-01",
        "search_budget_gpu_h": 18,
        "primary_metric": 0.806,
        "ci_low": 0.794,
        "ci_high": 0.818,
        "worst_slice_recall": 0.771,
        "p99_latency_ms": 39,
        "cost_per_1000": 0.48,
        "decision": "canary",
    },
]

required = {
    "model", "dataset", "search_budget_gpu_h", "primary_metric",
    "ci_low", "ci_high", "worst_slice_recall",
    "p99_latency_ms", "cost_per_1000", "decision",
}
table = pd.DataFrame(rows)
missing = required - set(table.columns)
if missing or table[list(required)].isna().any().any():
    raise ValueError(f"Decision evidence is incomplete: {sorted(missing)}")

table["interval"] = table.apply(
    lambda row: f"[{row.ci_low:.3f}, {row.ci_high:.3f}]", axis=1
)
table["feasible"] = (
    (table["worst_slice_recall"] >= 0.75)
    & (table["p99_latency_ms"] <= 50)
    & (table["cost_per_1000"] <= 0.50)
)

print(
    table[
        [
            "model", "dataset", "primary_metric", "interval",
            "worst_slice_recall", "p99_latency_ms",
            "cost_per_1000", "feasible", "decision",
        ]
    ].to_string(index=False)
)
```

</details>

表格暴露了一个重要矛盾：现有模型不满足新声明的切片要求，但书面决策仍写着“保留”。这个冲突应触发审核，而不是在格式化过程中被悄悄忽略。报告工具可以强制信息完整并检查算术；最终决策仍必须由负责任的人解决。

**对比总结。** 实验日志保存发生过的一切；技术报告选择支持主张所需的证据；运行手册规定应采取什么行动；模型卡或系统卡总结预期用途与局限。这些工件会有重叠，但服务于不同读者和决策。


### **综合研究工作流**

综合工作流把科学纪律与工程纪律结合起来。这里刻意使用**关卡**而不是线性清单：关卡失败会改变下一步行动。团队可能需要修改标签、收集数据、简化模型、重做实验、重新设计服务，或者停止项目。

<div class="diagram-scroll">

![综合工作流使用契约、证据、工件、发布和运行关卡。](assets/capstone-release-gates.svg){fig-alt="五个审核关卡覆盖决策契约、实验证据、可复现工件、安全发布和持续运行。"}

</div>

#### **关卡 1：决策契约**

在建模前写一页说明：

- 决策负责人和受影响用户；
- 单位、预测时刻、目标、时间范围与标签成熟条件；
- 行动策略、处理容量与回退方案；
- 当前基线和预期价值路径；
- 主要指标、护栏、关键切片和硬约束；
- 确定性或非 ML 方法不足的原因；
- 数据或可行动性不足时的停止条件。

**通过证据：**利益相关者同意目标与指标确实对应真实决策，并且项目有长期维护负责人。

#### **关卡 2：数据与实验证据**

构造带契约、血缘、时间点正确性、群组感知划分和标签覆盖分析的版本化数据集。实现虚拟、启发式和经典基线。在条件允许时，最终比较前预先登记候选协议：

- 候选家族与搜索预算；
- 验证和测试策略；
- 指标、阈值、切片与不确定性；
- 训练种子与停止规则；
- 资源测量；
- 消融、敏感性和失败分析。

**通过证据：**候选方案相对可信基线产生具有实际意义的提升，经受不确定性与关键切片检查，并且不违反离线约束。

#### **关卡 3：可复现工件**

制作可从干净环境运行的软件包：

- 不可变数据标识和划分分配；
- 代码 commit 与环境锁文件或容器 digest；
- 配置与随机种子；
- 一条训练命令和一条评估命令；
- 原始预测、指标、图表和资源日志；
- 已拟合预处理、模型签名与版本化模型；
- 决策报告、局限性与许可信息。

请另一位人员在干净环境中复现一张关键表。**通过证据：**无需未记录的人工干预，即可追踪并实质复现实验结果。

#### **关卡 4：发布**

针对服务契约和重放数据测试完整软件包，根据决策截止时间选择批量、在线或边缘部署。定义：

- 影子与金丝雀顺序；
- 最小样本量与晋升标准；
- 系统、数据、模型、切片和决策护栏；
- 遥测字段与模型版本日志；
- 降级、回滚与负责人；
- 安全和访问边界；
- 告警与事故手册。

**通过证据：**候选方案在代表性流量下满足 SLO 与护栏，并且团队已经实际演练回滚。

#### **关卡 5：运行、学习与退役**

持续监控输入契约、新鲜度、漂移、预测行为、延迟标签性能、决策结果、工作量与资源成本。定期审核：

- 事故历史与错误预算消耗；
- 反馈机制与选择性标签；
- 重训练触发、验证与晋升历史；
- 原始决策是否仍然相关；
- 更简单规则或更新流程是否已经支配当前模型；
- 数据保留、归档与退役义务。

**通过证据：**在当前契约下，继续运行仍能创造净价值。即使模型过去表现很好，只要已经没有负责人、可观测结果或安全回退方案，就应该退役。

<details>
<summary><strong>Python：把综合流程转化为显式发布关卡决策</strong></summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Evidence:
    decision_contract_signed: bool
    label_coverage: float
    point_in_time_audit_passed: bool
    baseline_gain: float
    lower_confidence_bound_gain: float
    worst_slice_recall: float
    reproducible_clean_run: bool
    p99_latency_ms: float
    availability: float
    rollback_tested: bool
    monitoring_owner_assigned: bool


def review(evidence: Evidence) -> dict:
    gates = {
        "contract": evidence.decision_contract_signed,
        "data": (
            evidence.label_coverage >= 0.95
            and evidence.point_in_time_audit_passed
        ),
        "experimental_evidence": (
            evidence.baseline_gain >= 0.02
            and evidence.lower_confidence_bound_gain > 0
            and evidence.worst_slice_recall >= 0.75
        ),
        "artifact": evidence.reproducible_clean_run,
        "release": (
            evidence.p99_latency_ms <= 50
            and evidence.availability >= 0.999
            and evidence.rollback_tested
            and evidence.monitoring_owner_assigned
        ),
    }
    first_failure = next((name for name, passed in gates.items() if not passed), None)
    return {
        "gates": gates,
        "decision": "APPROVE_CANARY" if first_failure is None else "STOP_AND_REVISE",
        "first_failed_gate": first_failure,
    }


candidate = Evidence(
    decision_contract_signed=True,
    label_coverage=0.982,
    point_in_time_audit_passed=True,
    baseline_gain=0.031,
    lower_confidence_bound_gain=0.012,
    worst_slice_recall=0.77,
    reproducible_clean_run=True,
    p99_latency_ms=43,
    availability=0.9995,
    rollback_tested=False,
    monitoring_owner_assigned=True,
)

print(review(candidate))
```

</details>

这个候选在发布关卡停止，因为尚未演练回滚。正确反应不是把 `False` 改成 `True`，而是执行回滚演练、记录证据，再重新审核关卡。

### **系统化总结**

| 问题 | 薄弱回答 | 强回答 |
|---|---|---|
| 我们在构建什么？ | “一个流失模型” | 包含单位、时间范围、行动、容量、用户和约束的决策契约 |
| 它是否更好？ | 分数高于一个未调优模型 | 相对可信基线的受控提升，并包含不确定性、切片和资源证据 |
| 它能否复现？ | 一个 Notebook 和模型文件 | 版本化数据、代码、环境、配置、预测与干净运行命令 |
| 它能否部署？ | 端点可以返回分数 | 软件包、schema、负载测试、安全发布、SLO、遥测、降级与回滚 |
| 它是否仍在正常工作？ | 仪表板看起来正常 | 覆盖系统、数据、模型、延迟结果和决策价值的有负责人告警 |
| 是否应该重训练？ | 某个特征发生漂移 | 持续证据触发候选训练；现有模型保留到发布关卡通过 |
| 研究是否可信？ | 标题数字可以匹配 | 主张、协议、预算、不确定性、消融、失败和可执行工件保持一致 |

研究与生产强调不同终点，却共享相同纪律。研究询问主张能否经受受控挑战与独立审查；生产询问决策系统能否在持续变化的运行环境中保持价值与可靠性。二者都需要精确范围、强基线、可追踪工件、不确定性、失败分析，以及对证据没有证明什么保持诚实。

机器学习系列在算法转化为负责任工作的地方结束：一项可以被检查的主张、一份可以被复现的工件、一次可以被撤回的发布，以及一个可以被观察、改进或退役的系统。
